# 03 - From boxes to animals: tracking

A detector gives boxes; a survey counts animals. Tracking is what turns one into the other - linking the boxes of the same animal across frames, so that fifty sightings become one individual. This notebook runs the framework's tracker on public data, shows *why* the pipeline tracks on the ground rather than in the image, imports tracks made elsewhere (TRex), and matches thermal tracks to RGB tracks by the clock the two cameras share.

Everything is arrays: `(N,)` frames, `(N, 4|6)` boxes in, `(N,)` track ids out - aligned with the input, nothing reordered or dropped. The files are read and written by `bambi.io.tracks`, byte-for-byte in the QGIS plugin's formats.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))
from _setup import ensure_environment, get_flight, get_dem, find
ensure_environment()

import json
import numpy as np
import matplotlib.pyplot as plt

## Part 1 - Flight 146: the annotators' tracks, re-derived

Flight 146 ships with `146_gt.txt`: 243 wild-boar boxes on 57 key frames, hand-linked into 12 tracks. That is a ground truth for tracking: throw the ids away, link the boxes again, and see how close the tracker gets. First the boxes go on the ground exactly as in `02_georeference`.

In [ ]:
from bambi.io.dem import read_dem_mesh
from bambi.io.poses import read_poses, to_local_poses
from bambi.io.corrections import read_corrections, corrections_for_frames
from bambi.io.tracks import read_mot
from bambi.geo.georef import boxes_to_world_by_frame, corners_to_extent

flight = get_flight("146", version="base")
dem = read_dem_mesh(get_dem("146"))
pf = read_poses(find(flight, "*_matched_poses.json")[0])
poses = to_local_poses(pf, epsg=dem.origin.epsg, origin=dem.origin)
t_corr, r_corr = corrections_for_frames(read_corrections(flight / "146_correction.json"), len(poses))

W = H = 1024
gt = read_mot(flight / "146_gt.txt")                          # frames, track_ids, boxes (xyxy), ...
truth = gt.track_ids
print(f"{len(gt)} boxes on {len(np.unique(gt.frames))} key frames, {len(np.unique(truth))} annotated tracks")

corners = boxes_to_world_by_frame(gt.frames, gt.boxes, poses, 50.0, W, H, dem.mesh, t_corr, r_corr)   # (N, 4, 3)
ground = corners_to_extent(corners)                            # (N, 6) [min_x min_y min_z max_x max_y max_z]
print("all on the DEM:", np.isfinite(ground).all())

### Link them again - in the image, and on the ground

`track_boxes` is the pipeline's built-in tracker: frame by frame, Hungarian assignment on box overlap (IoU), unmatched boxes open new tracks. It does not care what space the boxes live in, so it can be run twice on the same annotations: once on pixel boxes, once on their ground extents. The score is *purity*: for each track we produce, the share of its boxes that belong to the annotators' majority animal.

In [ ]:
from bambi.tracking.iou import track_boxes, interpolate_tracks

def purity(ids, truth):
    hits = sum(np.unique(truth[ids == t], return_counts=True)[1].max() for t in np.unique(ids))
    return hits / len(ids)

ids_px = track_boxes(gt.frames, gt.boxes, mode="hungarian", iou_threshold=0.3)
ids_gr = track_boxes(gt.frames, ground,   mode="hungarian", iou_threshold=0.3)
print(f"{'space':8s} {'tracks':>7s} {'purity':>7s}   (annotators: 12 tracks)")
print(f"{'pixels':8s} {len(np.unique(ids_px)):7d} {purity(ids_px, truth):7.2f}")
print(f"{'ground':8s} {len(np.unique(ids_gr)):7d} {purity(ids_gr, truth):7.2f}")

In pixels the tracker shatters: the drone flies on, so an animal that stands still drifts across the frame and stops overlapping itself between key frames. On the ground the same animal *stays where it is*, and the tracker links it. That is the reason the pipeline geo-references detections **before** tracking - and why the tracker takes `(N, 6)` DEM-local boxes as readily as pixel ones (overlap is computed on the horizontal extent).

Seen on the map, colour by our track id against the annotators' outline:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), sharex=True, sharey=True)
cx, cy = 0.5 * (ground[:, 0] + ground[:, 3]), 0.5 * (ground[:, 1] + ground[:, 4])
for ax, ids, title in ((axes[0], truth, "annotators' 12 tracks"), (axes[1], ids_gr, f"tracker on the ground: {len(np.unique(ids_gr))} tracks")):
    for t in np.unique(ids):
        m = ids == t
        o = np.argsort(gt.frames[m])
        ax.plot(cx[m][o], cy[m][o], "-o", ms=3, lw=1)
    ax.set_aspect("equal"); ax.set_title(title); ax.set_xlabel("east (m)")
axes[0].set_ylabel("north (m)"); plt.tight_layout(); plt.show()

### Fill the gaps, write the file

Key frames are 1-13 frames apart. `interpolate_tracks` closes every gap inside a track with linearly interpolated boxes, flags them, and keeps an index (`source`) back to the detection each row came from (`-1` for the invented ones). `write_tracks_csv` writes the plugin's `tracks.csv` - the same bytes the QGIS step produces from the same input.

In [ ]:
from bambi.io.tracks import write_tracks_csv, read_tracks_csv

tracks = interpolate_tracks(gt.frames, ids_gr, ground, gt.confidences, gt.classes)
print(f"{len(gt)} detections -> {len(tracks)} track rows, {tracks.interpolated.sum()} interpolated, {tracks.n_tracks} tracks")
out = flight / "146_tracks_demo.csv"
write_tracks_csv(out, tracks)
print(open(out).readline().strip(), "...")
back = read_tracks_csv(out)
print("round trip:", np.array_equal(back.track_ids, tracks.track_ids), "| the plugin's own writer produces the same text (see tests/test_parity_tracking_plugin.py)")

## Part 2 - Tracks made elsewhere: TRex

Not every project tracks with this pipeline. [TRex](https://trex.run) writes one `.npz` per individual with per-frame key-points on the *raw video*; the plugin's "Import TRex Tracklets" turns those into boxes and puts them on the ground. `bambi.io.trex` reads the files, `bambi.geo.calibration.undistort_boxes` moves raw-video pixels into the extracted-frame space the poses were built for, and `boxes_to_world_by_frame` does the rest.

There is no public TRex output for a BAMBI flight, so this writes a small tracklet folder from the annotations above (as if TRex had produced them) and reads it back:

In [ ]:
from bambi.io.trex import read_trex_tracklets

npz_dir = flight / "trex_demo"; npz_dir.mkdir(exist_ok=True)
all_frames = np.arange(gt.frames.min(), gt.frames.max() + 1)
for t in np.unique(truth):
    m = truth == t
    rows = {int(f): b for f, b in zip(gt.frames[m], gt.boxes[m])}
    px = np.full((len(all_frames), 4), np.nan); py = np.full((len(all_frames), 4), np.nan)     # 4 key-points: the corners
    for i, f in enumerate(all_frames):
        if int(f) in rows:
            x1, y1, x2, y2 = rows[int(f)]
            px[i] = [x1, x2, x2, x1]; py[i] = [y1, y1, y2, y2]
    np.savez(npz_dir / f"146_id{t}.npz", frame=all_frames.astype(np.float32), id=np.array([t], np.uint64),
             detection_p=np.ones(len(all_frames), np.float32), detection_class=np.zeros(len(all_frames), np.float32),
             video_size=np.array([1024.0, 1024.0]),
             **{f"poseX{k}": px[:, k].astype(np.float32) for k in range(4)},
             **{f"poseY{k}": py[:, k].astype(np.float32) for k in range(4)})

trex = read_trex_tracklets(npz_dir)
print(f"{len(trex.files)} files -> {len(trex)} rows, {len(np.unique(trex.track_ids))} individuals, video {trex.video_size}")
print("boxes recovered exactly:", np.allclose(np.sort(trex.boxes, axis=0), np.sort(gt.boxes, axis=0)))
# The processed video is already undistorted, so the boxes go straight to the ground; for raw DJI
# footage insert undistort_boxes(trex.boxes, mtx, dist, trex.video_size, extracted_frame_size) here.
trex_ground = corners_to_extent(boxes_to_world_by_frame(trex.frames, trex.boxes, poses, 50.0, W, H, dem.mesh, t_corr, r_corr))
print("ground extents identical to Part 1:", np.allclose(np.sort(trex_ground, axis=0), np.sort(ground, axis=0)))

## Part 3 - Two cameras, one animal: cross-modal matching

The M3T records thermal and RGB at once. A track confirmed in *both* is a real animal; the paper behind this (\"When One Modality Is Not Enough\", section 3.2) found that of 34 tracks seen in only one modality, one was real. `bambi.tracking.matching` does the confirmation in three steps: pair frames by the capture clock the two cameras share, fit an affine `RGB -> thermal` from co-occurring detections (bootstrapped - the correspondence is what we are solving for), then assign tracks one-to-one by the median inter-centre distance.

The public flight has thermal annotations only, so the RGB side here is **synthetic**: the thermal boxes pushed through a made-up lens relationship (zoom, small rotation, offset, jitter) with their own frame numbering and a clock 30 ms behind. The matcher does not know any of that.

In [ ]:
from bambi.tracking import matching as tm
from bambi.io.poses import epochs_from_timestamps

# thermal side: our tracks from Part 1 (pixel boxes, our track ids)
det_t = tm.detections(gt.frames, ids_gr, gt.boxes, gt.confidences)

# synthetic RGB side: a different lens, its own frame index (offset by 40) and its own clock
truth_affine = tm.Affine.from_coefficients(0.80, 0.05, -0.05, 0.80, 30.0, -12.0)      # RGB -> thermal
inverse = np.linalg.inv(truth_affine.matrix)
c_t = 0.5 * (gt.boxes[:, :2] + gt.boxes[:, 2:])
rng = np.random.default_rng(3)
c_w = (c_t - truth_affine.offset) @ inverse.T + rng.normal(0, 0.8, size=c_t.shape)
det_w = tm.detections(gt.frames + 40, truth, np.column_stack([c_w - 6, c_w + 6]), gt.confidences)  # annotators' ids on the RGB side

epochs_t = epochs_from_timestamps(pf.timestamps)                       # the SRT clock, per thermal frame
epochs_w = np.full(len(epochs_t) + 40, np.nan); epochs_w[40:] = epochs_t + 0.005   # RGB frame k+40 taken 5 ms after thermal frame k
frame_map = tm.match_frames_by_time(epochs_t, epochs_w, max_dt=0.1)      # thermal frame -> RGB frame or -1
pairs = tm.frame_pairs_from_map(frame_map)
print(f"{len(pairs)} thermal frames have an RGB partner within 100 ms; e.g. {pairs[2000].tolist()}")

result = tm.match_tracks(det_t, det_w, pairs, frame_size_t=(W, H), frame_size_w=(W, H))
print(f"affine recovered: {np.round(result.affine.coefficients, 3)}  (truth {truth_affine.coefficients})  RMSE {result.affine_rmse:.2f} px")
c = result.candidates
print(f"{result.n_candidates} candidate pairs, {len(result.matches)} confirmed:")
for i in result.matches:
    print(f"  thermal track {c.track_id_t[i]:3d} <-> RGB track {c.track_id_w[i]:5d}   shared {c.shared[i]:2d} frames, median {c.median_dist[i]:.1f} px")

Every confirmed pair is a thermal track of ours meeting the annotators' RGB track it was cut from; the tracks the gates refused (`tm.rejection_reasons`) are the short fragments Part 1 left. Confirmation is the point: what survives here is what a census can count.

## Where this goes next

`04_survey_analytics` takes ground tracks like these and turns them into perpendicular distances, densities and population estimates. Parity of every step in this notebook with the QGIS plugin is asserted in `tests/test_parity_tracking_plugin.py` (the tracker, on 48k real detections) and `tests/test_parity_trex_plugin.py` (a real TRex import).